# 02 — Fingerspelling CNN Bake-off (CRISP-DM: Modeling + Evaluation)

Trains **4 candidates under one fixed protocol** on the Kaggle ASL Alphabet (87k images, 29 classes),
then scores them and picks a winner. **Run on Colab with a GPU** (Runtime > Change runtime type > T4 GPU)
so every candidate uses the same batch size = a fair comparison.

Candidates: `cnn_scratch` (built from scratch) vs `mobilenetv2`, `efficientnetb0`, `resnet50` (ImageNet transfer).

## 1. Setup (run once)
Uploads your `kaggle.json` (Kaggle > Account > Create New API Token) and downloads the dataset (~1 GB).

In [ ]:
!git clone https://github.com/SpliiiiT/GROOPY.git
%cd GROOPY
!pip install -q kaggle   # TensorFlow / scikit-learn / matplotlib are already on Colab


In [ ]:
# Upload kaggle.json when prompted
from google.colab import files; files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!python data/download_asl_alphabet.py   # 87k images, ~1 GB

## 2. Confirm the GPU is visible

In [ ]:
import sys; sys.path.insert(0, '.')
import tensorflow as tf
print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

## 3. Train the whole bake-off
Same protocol for every model (fixed split, augmentation, batch size, 20 epochs; pretrained backbones do 2-phase fine-tuning).

In [ ]:
from recognition.src.train import train_one
from recognition.src.data import make_datasets
from recognition.src import models as zoo

train_ds, val_ds, test_ds, class_names = make_datasets()
records = [train_one(name, epochs=20, train_ds=train_ds, val_ds=val_ds) for name in zoo.REGISTRY]
records

## 4. Evaluate all models + weighted scorecard -> winner

In [ ]:
from recognition.src.evaluate import evaluate_model
from recognition.src import scorecard as sc
from recognition.src.config import MODELS_DIR
from pathlib import Path
import pandas as pd

rows = [evaluate_model(p.stem, p, test_ds, class_names)
        for p in sorted(Path(MODELS_DIR).glob('*.keras')) if not p.stem.startswith('word_')]
ranked = sc.score(rows)
pd.DataFrame(ranked)[['model','accuracy','macro_f1','latency','size','total']]

## 5. Grad-CAM (trust/bias check) + export the winner
1. `!python -m recognition.src.xai_gradcam --model recognition/models/<model>.keras` — heatmaps should sit on the HAND; update each model's **robustness** in the scorecard.
2. Add **stability** from the live desktop test, then re-run the scorecard cell.
3. Export the winner:


In [ ]:
winner = ranked[0]['model']; print('winner:', winner)
!python -m recognition.src.export --model recognition/models/{winner}.keras --target all
# download the exported TFLite/TF.js/keras from recognition/models/ (Files pane)